# Analysis

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import cohen_kappa_score, confusion_matrix, f1_score

from config import data_root, final_dataset, gold_final, llm_labels, models_dir

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

n_bootstrap = 1000
random_seed = 2026
few_shot_example_case_ids = {11, 374, 437, 455}

outcome_labels = ["afwijkend", "niet-afwijkend"]
outcome_unknown = ["afwijkend", "niet-afwijkend", "onbekend"]
yes_no = ["ja", "nee"]
yes_no_unknown = ["ja", "nee", "onbekend"]

gold_rename = {
    "definitief_handmatig_label": "handmatig_label",
    "definitief_artrose_aanwezig": "handmatig_artrose_aanwezig",
    "definitief_artrose_graad": "handmatig_artrose_graad",
    "definitief_nhg": "handmatig_nhg",
    "herkomst": "gold_herkomst",
}

gold_columns = [
    "case_id", "handmatig_label", "handmatig_artrose_aanwezig",
    "handmatig_artrose_graad", "handmatig_nhg",
    "label_martijn", "label_heleen",
    "label_artrose_martijn", "label_artrose_heleen",
    "artrose_graad_martijn", "artrose_graad_heleen",
    "nhg_martijn", "nhg_heleen", "gold_herkomst",
]

In [ ]:
def pct(n, total):
    return np.nan if total == 0 else 100 * n / total


def counts_table(series, order=None, total=None):
    counts = series.value_counts(dropna=False)
    if order is not None:
        counts = counts.reindex(order, fill_value=0)
    if total is None:
        total = int(counts.sum())
    return pd.DataFrame({
        "n": counts.astype(int),
        "%": [pct(v, total) for v in counts],
    }).round(1)


def cross_tables(df, row, col, row_order=None, col_order=None, valid_only=False):
    work = df.copy()
    if valid_only:
        work = work[work[row].isin(row_order) & work[col].isin(col_order)]
    counts = pd.crosstab(work[row], work[col])
    if row_order is not None:
        counts = counts.reindex(row_order, fill_value=0)
    if col_order is not None:
        counts = counts.reindex(columns=col_order, fill_value=0)
    row_pct = counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0) * 100
    return work, counts, row_pct.round(1)


def bootstrap_f1(y_true, y_pred, n=n_bootstrap, seed=random_seed):
    rng = np.random.default_rng(seed)
    scores = []
    idx = np.arange(len(y_true))
    for _ in range(n):
        sample = rng.choice(idx, size=len(idx), replace=True)
        yt = [y_true[i] for i in sample]
        yp = [y_pred[i] for i in sample]
        if len(set(yt)) >= 2:
            scores.append(f1_score(yt, yp, average="macro", zero_division=0))
    if not scores:
        return np.nan, np.nan
    return float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5))


def evaluate_pair(y_true, y_pred, pos="afwijkend", neg="niet-afwijkend"):
    pairs = [(t, p) for t, p in zip(y_true, y_pred) if t in (pos, neg) and p in (pos, neg)]
    if not pairs:
        return None
    yt = [x[0] for x in pairs]
    yp = [x[1] for x in pairs]
    cm = confusion_matrix(yt, yp, labels=[pos, neg])
    tp, fn, fp, tn = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
    lo, hi = bootstrap_f1(yt, yp)
    return {
        "n": len(yt),
        "accuracy": sum(a == b for a, b in zip(yt, yp)) / len(yt),
        "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
        "ci_low": lo,
        "ci_high": hi,
        "kappa": cohen_kappa_score(yt, yp) if len(set(yt)) > 1 else np.nan,
        "sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
        "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "tp": int(tp), "fn": int(fn), "fp": int(fp), "tn": int(tn),
    }


def metric_frame(rows):
    return pd.DataFrame(rows).round({
        "accuracy": 3, "macro_f1": 3, "ci_low": 3, "ci_high": 3,
        "kappa": 3, "sensitivity": 3, "specificity": 3,
    })


def merge_gold_with_primary(gold_source, primary):
    gold_base = gold_source.rename(columns=gold_rename)
    gold_base = gold_base[[c for c in gold_columns if c in gold_base.columns]].copy()
    primary_cols = [c for c in primary.columns if c == "case_id" or c not in gold_base.columns]
    return gold_base.merge(primary[primary_cols], on="case_id", how="left", validate="one_to_one")


def subset_for_variant(df, variant):
    if variant == "few-shot" and "case_id" in df.columns:
        return df[~df["case_id"].isin(few_shot_example_case_ids)].copy()
    return df


def latex_int(n):
    return f"{int(n):,}".replace(",", "{,}")


def latex_count_pct(n, total):
    if total == 0:
        return "0 (n/a)"
    return f"{latex_int(n)} ({100 * n / total:.1f}\\%)"


def latex_rows(counts, row_names, cols, title="Primary dataset, zero-shot"):
    total = int(counts.values.sum())
    rows = [f"\\multicolumn{{5}}{{l}}{{{title} ($n = {latex_int(total)}$)}} \\\\"]
    for row in counts.index:
        row_total = int(counts.loc[row].sum())
        first = int(counts.loc[row, cols[0]]) if cols[0] in counts.columns else 0
        second = int(counts.loc[row, cols[1]]) if cols[1] in counts.columns else 0
        rows.append(
            f"{row_names.get(row, str(row)):<14s} & "
            f"{latex_count_pct(first, row_total)} & "
            f"{latex_count_pct(second, row_total)} & "
            f"{latex_int(row_total)} & {pct(row_total, total):.1f} \\\\"
        )
    return pd.DataFrame({"latex": rows})


def show_table(name):
    item = tables[name]
    if isinstance(item, dict):
        for key, value in item.items():
            display(Markdown(f"**{key}**"))
            display(value)
    else:
        display(item)

## Load data

In [ ]:
primary = pd.read_excel(final_dataset)
llm_full = pd.read_excel(llm_labels)
gold_source = pd.read_excel(gold_final)
gold = merge_gold_with_primary(gold_source, primary)

n_primary = len(primary)
n_gold = len(gold)

tables = {}
tables["dataset_overview"] = pd.DataFrame([
    {"dataset": "full llm dataset", "rows": len(llm_full), "unique_cases": llm_full["case_id"].nunique()},
    {"dataset": "primary dataset", "rows": n_primary, "unique_cases": primary["case_id"].nunique()},
    {"dataset": "gold standard", "rows": n_gold, "unique_cases": gold["case_id"].nunique()},
])
display(tables["dataset_overview"])

## 1. Dataset overview

In [ ]:
tables["dataset_distributions"] = {
    "laterality": counts_table(llm_full["zijde"], total=len(llm_full)),
    "is_split": counts_table(llm_full["is_split"], total=len(llm_full)),
    "label_protesis": counts_table(llm_full["label_protesis"], total=len(llm_full)),
}
show_table("dataset_distributions")

## 2. Exclusions

In [ ]:
excl_cols = [c for c in llm_full.columns if c.startswith("excl_")]

tables["exclusions"] = pd.DataFrame([
    {"criterion": col, "rows": int(llm_full[col].sum()), "%": pct(int(llm_full[col].sum()), len(llm_full))}
    for col in excl_cols
]).round(1)

rows = []
for col in excl_cols:
    mask = llm_full[col].astype(bool)
    n_group = int(mask.sum())
    if n_group == 0:
        continue
    n_afw = int((llm_full.loc[mask, "label_gecombineerd"] == "afwijkend").sum())
    rows.append({"criterion": col, "n_excl": n_group, "n_afwijkend": n_afw, "%_afwijkend": pct(n_afw, n_group)})
tables["exclusion_abnormal_rate"] = pd.DataFrame(rows).round(1)

section_variant_cols = [
    ("fnd-ZS", "label_bevindingen"),
    ("fnd-FS", "label_bevindingen_fewshot"),
    ("fnd-CS", "label_bevindingen_consensus"),
    ("con-ZS", "label_conclusie"),
    ("con-FS", "label_conclusie_fewshot"),
    ("con-CS", "label_conclusie_consensus"),
    ("cmb-ZS", "label_gecombineerd"),
    ("cmb-FS", "label_gecombineerd_fewshot"),
    ("cmb-CS", "label_gecombineerd_consensus"),
]

rows = []
for col in excl_cols:
    mask = llm_full[col].astype(bool)
    row = {"group": col, "n": int(mask.sum())}
    if row["n"] == 0:
        continue
    for short, label_col in section_variant_cols:
        labels = llm_full.loc[mask, label_col]
        valid = labels.isin(outcome_labels)
        row[short] = pct((labels[valid] == "afwijkend").sum(), valid.sum())
    rows.append(row)
tables["exclusion_by_section_variant"] = pd.DataFrame(rows).round(1)

show_table("exclusions")
display(Markdown("**abnormal rate per exclusion group - zero-shot combined**"))
display(tables["exclusion_abnormal_rate"])
display(Markdown("**abnormal rate by section and prompt variant**"))
display(tables["exclusion_by_section_variant"])

## 3. Gold labels

In [ ]:
tables["gold_distributions"] = {
    "radiological outcome": counts_table(gold["handmatig_label"], total=n_gold),
    "osteoarthritis": counts_table(gold["handmatig_artrose_aanwezig"], total=n_gold),
    "nhg": counts_table(gold["handmatig_nhg"], total=n_gold),
    "kl grade": counts_table(gold["handmatig_artrose_graad"], total=n_gold),
}
if "gold_herkomst" in gold.columns:
    tables["gold_distributions"]["origin"] = counts_table(gold["gold_herkomst"], total=n_gold)

distribution_rows = []
for variable, table in tables["gold_distributions"].items():
    tmp = table.reset_index().rename(columns={table.index.name or "index": "label"})
    tmp.insert(0, "variable", variable)
    distribution_rows.append(tmp)
tables["gold_distributions_combined"] = pd.concat(distribution_rows, ignore_index=True)

rows = []
sub = gold.dropna(subset=["label_martijn", "label_heleen"])
sub = sub[sub["label_martijn"].isin(outcome_labels) & sub["label_heleen"].isin(outcome_labels)]
if len(sub):
    rows.append({
        "task": "outcome", "n": len(sub),
        "agreement": (sub["label_martijn"] == sub["label_heleen"]).mean(),
        "kappa": cohen_kappa_score(sub["label_martijn"], sub["label_heleen"]),
        "expert_1_positive": int((sub["label_martijn"] == "afwijkend").sum()),
        "expert_1_positive_%": pct((sub["label_martijn"] == "afwijkend").sum(), len(sub)),
        "expert_2_positive": int((sub["label_heleen"] == "afwijkend").sum()),
        "expert_2_positive_%": pct((sub["label_heleen"] == "afwijkend").sum(), len(sub)),
        "disagreements": int((sub["label_martijn"] != sub["label_heleen"]).sum()),
    })

sub = gold.dropna(subset=["nhg_martijn", "nhg_heleen"])
sub = sub[sub["nhg_martijn"].isin(yes_no) & sub["nhg_heleen"].isin(yes_no)]
if len(sub):
    rows.append({"task": "nhg", "n": len(sub), "agreement": (sub["nhg_martijn"] == sub["nhg_heleen"]).mean(), "kappa": cohen_kappa_score(sub["nhg_martijn"], sub["nhg_heleen"])})

sub = gold.dropna(subset=["label_artrose_martijn", "label_artrose_heleen"])
sub = sub[sub["label_artrose_martijn"].isin(yes_no) & sub["label_artrose_heleen"].isin(yes_no)]
if len(sub):
    rows.append({"task": "osteoarthritis", "n": len(sub), "agreement": (sub["label_artrose_martijn"] == sub["label_artrose_heleen"]).mean(), "kappa": cohen_kappa_score(sub["label_artrose_martijn"], sub["label_artrose_heleen"])})

sub = gold.dropna(subset=["artrose_graad_martijn", "artrose_graad_heleen"])
sub = sub[pd.to_numeric(sub["artrose_graad_martijn"], errors="coerce").notna() & pd.to_numeric(sub["artrose_graad_heleen"], errors="coerce").notna()]
sub = sub[sub["handmatig_artrose_aanwezig"] == "ja"]
sub = sub[~sub["case_id"].isin([1, 2])]
if len(sub):
    y1 = pd.to_numeric(sub["artrose_graad_martijn"]).astype(int)
    y2 = pd.to_numeric(sub["artrose_graad_heleen"]).astype(int)
    rows.append({"task": "kl grade", "n": len(sub), "agreement": (y1 == y2).mean(), "kappa": cohen_kappa_score(y1, y2), "kappa_quadratic": cohen_kappa_score(y1, y2, weights="quadratic")})

tables["inter_rater"] = pd.DataFrame(rows).round(3)

display(tables["gold_distributions_combined"])
display(Markdown("**inter-rater agreement**"))
display(tables["inter_rater"])

## 4. LLM vs gold

In [ ]:
variants = [
    ("zero-shot", "label_gecombineerd"),
    ("few-shot", "label_gecombineerd_fewshot"),
    ("consensus", "label_gecombineerd_consensus"),
]

rows = []
for variant, col in variants:
    eval_gold = subset_for_variant(gold, variant)
    r = evaluate_pair(eval_gold["handmatig_label"], eval_gold[col])
    if r:
        rows.append({"variant": variant, "section": "combined", **r})
tables["llm_combined"] = metric_frame(rows)

rows = []
for variant, suffix in [("zero-shot", ""), ("few-shot", "_fewshot"), ("consensus", "_consensus")]:
    eval_gold = subset_for_variant(gold, variant)
    for section in ["bevindingen", "conclusie", "gecombineerd"]:
        col = f"label_{section}{suffix}"
        r = evaluate_pair(eval_gold["handmatig_label"], eval_gold[col])
        if r:
            rows.append({"variant": variant, "section": section, **r})
tables["llm_by_section"] = metric_frame(rows)

confusions = {}
for variant, col in variants:
    eval_gold = subset_for_variant(gold, variant)
    sub = eval_gold[eval_gold["handmatig_label"].isin(outcome_labels) & eval_gold[col].isin(outcome_labels)]
    confusions[variant] = pd.crosstab(sub["handmatig_label"], sub[col], rownames=["gold"], colnames=[variant])
tables["llm_confusions"] = confusions

rows = []
for variant, col in variants:
    eval_gold = subset_for_variant(gold, variant)
    r = evaluate_pair(eval_gold["handmatig_label"], eval_gold[col])
    if r:
        n_afw = r["tp"] + r["fn"]
        n_niet = r["tn"] + r["fp"]
        rows.append({
            "variant": variant,
            "afwijkend_correct": f"{r['tp']}/{n_afw}",
            "afwijkend_%": pct(r["tp"], n_afw),
            "niet_afwijkend_correct": f"{r['tn']}/{n_niet}",
            "niet_afwijkend_%": pct(r["tn"], n_niet),
        })
tables["llm_per_label_correct"] = pd.DataFrame(rows).round(1)

rows = []
for variant, suffix in [("zero-shot", ""), ("few-shot", "_fewshot"), ("consensus", "_consensus")]:
    eval_gold = subset_for_variant(gold, variant)
    for section in ["bevindingen", "conclusie", "gecombineerd"]:
        col = f"label_{section}{suffix}"
        if col not in eval_gold.columns:
            continue
        r = evaluate_pair(eval_gold["handmatig_label"], eval_gold[col])
        if r:
            n_afw = r["tp"] + r["fn"]
            n_niet = r["tn"] + r["fp"]
            rows.append({
                "variant": variant,
                "section": section,
                "afwijkend_correct": f"{r['tp']}/{n_afw}",
                "afwijkend_%": pct(r["tp"], n_afw),
                "niet_afwijkend_correct": f"{r['tn']}/{n_niet}",
                "niet_afwijkend_%": pct(r["tn"], n_niet),
            })
tables["llm_per_label_correct_by_section"] = pd.DataFrame(rows).round(1)

display(tables["llm_combined"])
display(Markdown("**by section**"))
display(tables["llm_by_section"])
show_table("llm_confusions")
display(Markdown("**per-label correct rates**"))
display(tables["llm_per_label_correct"])
display(Markdown("**per-label correct rates by section**"))
display(tables["llm_per_label_correct_by_section"])

## 5. NHG

In [ ]:
nhg_variants = [
    ("zero-shot", "label_nhg"),
    ("few-shot", "label_nhg_fewshot"),
    ("consensus", "label_nhg_consensus"),
]

rows = []
for variant, col in nhg_variants:
    sub = gold.dropna(subset=["handmatig_nhg", col])
    sub = sub[sub["handmatig_nhg"].isin(yes_no) & sub[col].isin(yes_no)]
    r = evaluate_pair(sub["handmatig_nhg"], sub[col], pos="ja", neg="nee")
    if r:
        rows.append({"variant": variant, **r})
tables["nhg_eval"] = metric_frame(rows)

tables["nhg_confusions"] = {}
for variant, col in nhg_variants:
    sub = gold.dropna(subset=["handmatig_nhg", col])
    sub = sub[sub["handmatig_nhg"].isin(yes_no) & sub[col].isin(yes_no)]
    tables["nhg_confusions"][variant] = pd.crosstab(sub["handmatig_nhg"], sub[col], rownames=["manual"], colnames=[variant])

rows = []
for label in yes_no_unknown:
    row = {"label": label}
    for variant, col in [("zero-shot", "label_nhg"), ("few-shot", "label_nhg_fewshot"), ("consensus", "label_nhg_consensus")]:
        row[variant] = int((primary[col] == label).sum())
        row[f"{variant}_%"] = pct(row[variant], n_primary)
    rows.append(row)
tables["nhg_distribution_primary"] = pd.DataFrame(rows).round(1)

def add_cross_table(key, source, row, col, row_order, col_order, valid_only=True):
    sub, counts, row_pct = cross_tables(source, row, col, row_order, col_order, valid_only=valid_only)
    tables[key] = {"counts": counts, "row_%": row_pct, "n": pd.DataFrame({"n": [len(sub)]})}
    return sub, counts, row_pct

nhg_names = {"ja": "Conforming", "nee": "Non-conforming"}
add_cross_table("nhg_x_outcome_zero_shot", primary, "label_nhg", "label_gecombineerd", yes_no, outcome_labels)
add_cross_table("nhg_x_outcome_consensus", primary, "label_nhg_consensus", "label_gecombineerd_consensus", yes_no, outcome_labels)
tables["nhg_x_outcome_zero_shot_latex"] = latex_rows(tables["nhg_x_outcome_zero_shot"]["counts"], nhg_names, outcome_labels)

trauma_order = ["trauma", "oud_trauma", "niet_trauma", "onbekend"]
fracture_order = ["ja", "nee", "onbekend"]
add_cross_table("trauma_x_nhg_zero_shot", primary, "label_trauma", "label_nhg", trauma_order, yes_no)
add_cross_table("trauma_x_nhg_consensus", primary, "label_trauma", "label_nhg_consensus", trauma_order, yes_no)
add_cross_table("fracture_x_nhg_zero_shot", primary, "label_fractuur", "label_nhg", fracture_order, yes_no)
add_cross_table("fracture_x_nhg_consensus", primary, "label_fractuur", "label_nhg_consensus", fracture_order, yes_no)
add_cross_table("nhg_x_osteoarthritis_zero_shot", primary, "label_nhg", "artrose_radiologie", yes_no, yes_no)
add_cross_table("nhg_x_osteoarthritis_consensus", primary, "label_nhg_consensus", "artrose_radiologie", yes_no, yes_no)
add_cross_table("nhg_x_outcome_gold", gold, "handmatig_nhg", "handmatig_label", yes_no, outcome_labels)
add_cross_table("nhg_consensus_x_outcome_gold", gold, "label_nhg_consensus", "handmatig_label", yes_no, outcome_labels)

display(tables["nhg_eval"])
display(Markdown("**primary dataset NHG labels**"))
display(tables["nhg_distribution_primary"])
for key in [
    "nhg_x_outcome_zero_shot", "nhg_x_outcome_consensus",
    "trauma_x_nhg_zero_shot", "trauma_x_nhg_consensus",
    "fracture_x_nhg_zero_shot", "fracture_x_nhg_consensus",
    "nhg_x_osteoarthritis_zero_shot", "nhg_x_osteoarthritis_consensus",
    "nhg_x_outcome_gold", "nhg_consensus_x_outcome_gold",
]:
    display(Markdown(f"**{key}**"))
    show_table(key)
display(Markdown("**LaTeX rows: NHG x outcome, zero-shot**"))
display(tables["nhg_x_outcome_zero_shot_latex"])

## 6. Osteoarthritis and KL

In [ ]:
sub = gold.dropna(subset=["artrose_radiologie", "handmatig_artrose_aanwezig"])
sub = sub[sub["artrose_radiologie"].isin(yes_no) & sub["handmatig_artrose_aanwezig"].isin(yes_no)]
tables["osteoarthritis_regex_vs_manual"] = metric_frame([evaluate_pair(sub["handmatig_artrose_aanwezig"], sub["artrose_radiologie"], pos="ja", neg="nee")])
tables["osteoarthritis_regex_confusion"] = pd.crosstab(sub["handmatig_artrose_aanwezig"], sub["artrose_radiologie"], rownames=["manual"], colnames=["regex"])

add_cross_table("osteoarthritis_referral_x_radiology", primary, "artrose_vraagstelling", "artrose_radiologie", yes_no, yes_no)

kl_summary = []
kl_confusions = {}
kl_variants = [
    ("zero-shot", "artrose_graad"),
    ("few-shot", "artrose_graad_fewshot"),
    ("consensus", "artrose_graad_consensus"),
]
for variant, col in kl_variants:
    sub = gold.dropna(subset=["handmatig_artrose_graad", col])
    sub = sub[pd.to_numeric(sub["handmatig_artrose_graad"], errors="coerce").notna() & pd.to_numeric(sub[col], errors="coerce").notna()]
    yt = pd.to_numeric(sub["handmatig_artrose_graad"]).astype(int)
    yp = pd.to_numeric(sub[col]).astype(int)
    kl_summary.append({
        "variant": variant,
        "n": len(sub),
        "exact": (yt == yp).mean(),
        "within_1": (abs(yt - yp) <= 1).mean(),
        "kappa_unweighted": cohen_kappa_score(yt, yp),
        "kappa_quadratic": cohen_kappa_score(yt, yp, weights="quadratic"),
    })
    kl_confusions[variant] = pd.crosstab(yt.rename("gold"), yp.rename(variant))
tables["kl_summary"] = pd.DataFrame(kl_summary).round(3)
tables["kl_confusions"] = kl_confusions

add_cross_table("osteoarthritis_x_outcome_gold", gold, "handmatig_artrose_aanwezig", "handmatig_label", yes_no, outcome_labels, valid_only=False)
add_cross_table("osteoarthritis_x_nhg_gold", gold, "handmatig_artrose_aanwezig", "handmatig_nhg", yes_no, yes_no_unknown, valid_only=False)
add_cross_table("osteoarthritis_x_consensus_outcome", primary, "artrose_radiologie", "label_gecombineerd_consensus", yes_no, outcome_unknown, valid_only=False)
tables["artrose_radiologie_distribution_primary"] = counts_table(primary["artrose_radiologie"], order=yes_no, total=n_primary)

for key in [
    "osteoarthritis_regex_vs_manual", "osteoarthritis_regex_confusion",
    "osteoarthritis_referral_x_radiology", "kl_summary", "kl_confusions",
    "osteoarthritis_x_outcome_gold", "osteoarthritis_x_nhg_gold",
    "osteoarthritis_x_consensus_outcome", "artrose_radiologie_distribution_primary",
]:
    display(Markdown(f"**{key}**"))
    show_table(key)

## 7. Primary dataset cross-tabs

In [ ]:
primary_cross_specs = [
    ("trauma_x_outcome_zero_shot", "label_trauma", "label_gecombineerd", trauma_order, outcome_unknown),
    ("trauma_x_outcome_consensus", "label_trauma", "label_gecombineerd_consensus", trauma_order, outcome_unknown),
    ("osteoarthritis_referral_x_outcome_zero_shot", "artrose_vraagstelling", "label_gecombineerd", yes_no, outcome_unknown),
    ("osteoarthritis_referral_x_outcome_consensus", "artrose_vraagstelling", "label_gecombineerd_consensus", yes_no, outcome_unknown),
    ("fracture_x_outcome_zero_shot", "label_fractuur", "label_gecombineerd", fracture_order, outcome_unknown),
    ("fracture_x_outcome_consensus", "label_fractuur", "label_gecombineerd_consensus", fracture_order, outcome_unknown),
    ("trauma_x_osteoarthritis_referral", "label_trauma", "artrose_vraagstelling", trauma_order, yes_no),
    ("trauma_x_osteoarthritis_radiology", "label_trauma", "artrose_radiologie", trauma_order, yes_no),
    ("osteoarthritis_referral_x_nhg_zero_shot", "artrose_vraagstelling", "label_nhg", yes_no, yes_no_unknown),
    ("osteoarthritis_referral_x_nhg_consensus", "artrose_vraagstelling", "label_nhg_consensus", yes_no, yes_no_unknown),
]
for key, row, col, row_order, col_order in primary_cross_specs:
    add_cross_table(key, primary, row, col, row_order, col_order, valid_only=False)
    display(Markdown(f"**{key}**"))
    show_table(key)

sub_aq = primary[primary["artrose_vraagstelling"] == "ja"]
gold_art = gold[gold["handmatig_artrose_aanwezig"].isin(["ja"]) & gold["handmatig_label"].isin(outcome_labels)]
tables["arthrosis_comparison"] = pd.DataFrame([
    {"group": "arthrosis in referral question", "dataset": "primary", "n": len(sub_aq), "% abnormal": pct((sub_aq["label_gecombineerd_consensus"] == "afwijkend").sum(), len(sub_aq))},
    {"group": "arthrosis present in radiology report", "dataset": "manual gold", "n": len(gold_art), "% abnormal": pct((gold_art["handmatig_label"] == "afwijkend").sum(), len(gold_art))},
]).round(1)
display(Markdown("**arthrosis comparison**"))
display(tables["arthrosis_comparison"])

## 8. Variant agreement

In [ ]:
rows = []
for a, b, col_a, col_b in [
    ("zero-shot", "few-shot", "label_gecombineerd", "label_gecombineerd_fewshot"),
    ("zero-shot", "consensus", "label_gecombineerd", "label_gecombineerd_consensus"),
    ("few-shot", "consensus", "label_gecombineerd_fewshot", "label_gecombineerd_consensus"),
]:
    sub = primary.dropna(subset=[col_a, col_b])
    agree = int((sub[col_a] == sub[col_b]).sum())
    rows.append({"variant_a": a, "variant_b": b, "n": len(sub), "agree": agree, "agree_%": pct(agree, len(sub)), "kappa": cohen_kappa_score(sub[col_a], sub[col_b])})
tables["variant_agreement"] = pd.DataFrame(rows).round(3)

rows = []
for a, b, col_a, col_b in [
    ("zero-shot nhg", "few-shot nhg", "label_nhg", "label_nhg_fewshot"),
    ("zero-shot nhg", "consensus nhg", "label_nhg", "label_nhg_consensus"),
    ("few-shot nhg", "consensus nhg", "label_nhg_fewshot", "label_nhg_consensus"),
]:
    sub = primary.dropna(subset=[col_a, col_b])
    agree = int((sub[col_a] == sub[col_b]).sum())
    rows.append({"variant_a": a, "variant_b": b, "n": len(sub), "agree": agree, "agree_%": pct(agree, len(sub)), "kappa": cohen_kappa_score(sub[col_a], sub[col_b])})
tables["nhg_variant_agreement"] = pd.DataFrame(rows).round(3)

display(tables["variant_agreement"])
display(Markdown("**NHG**"))
display(tables["nhg_variant_agreement"])

## 9. Label distributions

In [ ]:
rows = []
for section in ["bevindingen", "conclusie", "gecombineerd"]:
    for label in outcome_unknown:
        row = {"section": section, "label": label}
        for variant, suffix in [("zero-shot", ""), ("few-shot", "_fewshot"), ("consensus", "_consensus")]:
            col = f"label_{section}{suffix}"
            row[variant] = int((primary[col] == label).sum())
            row[f"{variant}_%"] = pct(row[variant], n_primary)
        rows.append(row)
tables["llm_label_distribution_primary"] = pd.DataFrame(rows).round(1)

rows = []
for grade in [0, 1, 2, 3, 4, "onbekend"]:
    row = {"grade": grade}
    for variant, col in [("zero-shot", "artrose_graad"), ("few-shot", "artrose_graad_fewshot"), ("consensus", "artrose_graad_consensus")]:
        row[variant] = int((primary[col].astype(str) == str(grade)).sum())
        row[f"{variant}_%"] = pct(row[variant], n_primary)
    rows.append(row)
tables["kl_label_distribution_primary"] = pd.DataFrame(rows).round(1)

rows = []
for label in yes_no_unknown:
    row = {"label": label}
    for variant, col in [("zero-shot", "label_nhg"), ("few-shot", "label_nhg_fewshot"), ("consensus", "label_nhg_consensus")]:
        row[variant] = int((primary[col] == label).sum())
        row[f"{variant}_%"] = pct(row[variant], n_primary)
    rows.append(row)
tables["nhg_label_distribution_primary"] = pd.DataFrame(rows).round(1)

display(tables["llm_label_distribution_primary"])
display(Markdown("**KL**"))
display(tables["kl_label_distribution_primary"])
display(Markdown("**NHG**"))
display(tables["nhg_label_distribution_primary"])

## 10. Findings vs conclusion

In [ ]:
rows = []
for variant, suffix in [("zero-shot", ""), ("consensus", "_consensus")]:
    col_b = f"label_bevindingen{suffix}"
    col_c = f"label_conclusie{suffix}"
    sub = primary.dropna(subset=[col_b, col_c])
    agree = int((sub[col_b] == sub[col_c]).sum())
    rows.append({"variant": variant, "n": len(sub), "agree": agree, "agree_%": pct(agree, len(sub)), "kappa": cohen_kappa_score(sub[col_b], sub[col_c])})
tables["findings_conclusion_agreement"] = pd.DataFrame(rows).round(3)

sub = primary.dropna(subset=["label_bevindingen", "label_conclusie"])
tables["findings_conclusion_disagreements"] = (
    sub[sub["label_bevindingen"] != sub["label_conclusie"]]
    .groupby(["label_bevindingen", "label_conclusie"])
    .size()
    .sort_values(ascending=False)
    .head(6)
    .reset_index(name="n")
)

display(tables["findings_conclusion_agreement"])
display(Markdown("**most common disagreements, zero-shot**"))
display(tables["findings_conclusion_disagreements"])

## 11. Confidence

In [ ]:
conf_cols = ["conf_afwijkend_gecombineerd", "conf_niet_afwijkend_gecombineerd", "conf_onbekend_gecombineerd"]
avail_conf = [c for c in conf_cols if c in primary.columns]
conf_vals = primary[avail_conf].max(axis=1).dropna()

tables["confidence_summary"] = pd.DataFrame([{
    "n": len(conf_vals),
    "mean": conf_vals.mean(),
    "median": conf_vals.median(),
    "min": conf_vals.min(),
}]).round(3)
tables["confidence_thresholds"] = pd.DataFrame([
    {"threshold": t, "n": int((conf_vals >= t).sum()), "%": pct((conf_vals >= t).sum(), len(conf_vals))}
    for t in [0.95, 0.90, 0.80, 0.70]
]).round(1)

gold_conf = gold[gold["handmatig_label"].isin(outcome_labels) & gold["label_gecombineerd"].isin(outcome_labels)].copy()
avail2 = [c for c in avail_conf if c in gold_conf.columns]
if avail2:
    gold_conf["correct"] = gold_conf["handmatig_label"] == gold_conf["label_gecombineerd"]
    gold_conf["conf_max"] = gold_conf[avail2].max(axis=1)
    tables["confidence_correct_vs_incorrect"] = pd.DataFrame([
        {"group": "correct", "mean_confidence": gold_conf[gold_conf["correct"]]["conf_max"].mean()},
        {"group": "incorrect", "mean_confidence": gold_conf[~gold_conf["correct"]]["conf_max"].mean()},
    ]).round(3)
    rows = []
    for lo, hi in [(0.0, 0.5), (0.5, 0.7), (0.7, 0.8), (0.8, 0.9), (0.9, 0.95), (0.95, 1.001)]:
        in_bin = gold_conf[(gold_conf["conf_max"] >= lo) & (gold_conf["conf_max"] < hi)]
        if len(in_bin):
            correct = int(in_bin["correct"].sum())
            rows.append({"bin": f"{lo:.2f}-{min(hi, 1):.2f}", "n_total": len(in_bin), "n_correct": correct, "n_error": len(in_bin) - correct, "%_correct": pct(correct, len(in_bin)), "%_error": pct(len(in_bin) - correct, len(in_bin))})
    tables["confidence_bins"] = pd.DataFrame(rows).round(1)
    try:
        from scipy.stats import pearsonr
        r, p = pearsonr(gold_conf["conf_max"].fillna(0), gold_conf["correct"].astype(float))
        tables["confidence_correlation"] = pd.DataFrame([{"pearson_r": r, "p": p}]).round(4)
    except ImportError:
        tables["confidence_correlation"] = pd.DataFrame()

for key in ["confidence_summary", "confidence_thresholds", "confidence_correct_vs_incorrect", "confidence_bins", "confidence_correlation"]:
    display(Markdown(f"**{key}**"))
    display(tables[key])

## 12. Summary

In [ ]:
tables["summary_llm_combined"] = tables["llm_combined"].copy()
tables["summary_by_section"] = tables["llm_by_section"].copy()
tables["summary_nhg"] = tables["nhg_eval"].copy()
tables["summary_kl"] = tables["kl_summary"].copy()
tables["summary_dataset"] = tables["dataset_overview"].copy()

for key in ["summary_llm_combined", "summary_by_section", "summary_nhg", "summary_kl", "summary_dataset"]:
    display(Markdown(f"**{key}**"))
    display(tables[key])

## 13. Prediction model results

In [ ]:
def read_json(path):
    path = Path(path)
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


rows = []
for model, fname in [
    ("majority baseline", "majority_baseline_minwords0_gold_test_metrics.json"),
    ("tfidf log.regression", "tfidf_logreg_minwords0_gold_test_metrics.json"),
    ("tfidf MLP", "tfidf_mlp_minwords0_gold_test_metrics.json"),
    ("tfidf XGBoost", "tfidf_xgboost_minwords0_gold_test_metrics.json"),
]:
    m = read_json(models_dir / fname)
    if not m:
        continue
    sens = m.get("sensitivity_afwijkend", {}).get("value")
    spec = m.get("specificity_non_abnormal", {}).get("value")
    rows.append({
        "model": model,
        "n": m.get("n"),
        "accuracy": m.get("accuracy", {}).get("value"),
        "macro_f1": m.get("macro_f1", {}).get("value"),
        "afwijkend_correct_%": 100 * sens if sens is not None else np.nan,
        "niet_afwijkend_correct_%": 100 * spec if spec is not None else np.nan,
    })
tables["classical_model_results"] = pd.DataFrame(rows).round(3)

rows = []
for model, path in [
    ("GP only", models_dir / "bert" / "gp_only"),
    ("GP + radiology text", models_dir / "bert" / "gp_plus_radiology"),
]:
    m = read_json(path / "metrics.json")
    if not m:
        continue
    rows.append({
        "model": model,
        "n": m.get("n_gold"),
        "accuracy": m.get("accuracy"),
        "macro_f1": m.get("macro_f1"),
        "afwijkend_correct_%": 100 * m.get("sensitivity_afwijkend") if m.get("sensitivity_afwijkend") is not None else np.nan,
        "niet_afwijkend_correct_%": 100 * m.get("specificity_niet_afwijkend") if m.get("specificity_niet_afwijkend") is not None else np.nan,
    })
tables["bert_model_results"] = pd.DataFrame(rows).round(3)

display(tables["classical_model_results"])
display(Markdown("**BERT**"))
display(tables["bert_model_results"])

## Table list

In [ ]:
pd.DataFrame({"table": list(tables)})